In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# Target URL
base_url = "https://warehouse-theme-metal.myshopify.com/collections/audio"
headers = {"User-Agent": "Mozilla/5.0"}

all_products = []
page = 1

while True:
    url = f"{base_url}?page={page}"
    print("Fetching:", url)

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print("Stopped: Invalid response.")
        break

    soup = BeautifulSoup(response.text, "html.parser")

    # Shopify Warehouse Theme product card selector
    product_cards = soup.select("div.product-item")

    if not product_cards:
        print("No more products found. Stopping.")
        break

    for card in product_cards:

        # ===== NAME =====
        name_elem = card.select_one(".product-item__title")
        name = name_elem.get_text(strip=True) if name_elem else "N/A"

        # ===== PRICE =====
        # Selectors for current price (could be sale or regular)
        price_elem = (
            card.select_one(".price.price--highlight") or # For sale price
            card.select_one(".price") # For regular price
        )

        price_raw = price_elem.get_text(strip=True) if price_elem else "N/A"

        # Clean price
        price_clean = "N/A"
        if price_raw != "N/A":
            num = re.sub(r"[^0-9.]", "", price_raw)
            try:
                price_clean = float(num)
            except:
                price_clean = "N/A"

        # ===== COMPARE PRICE =====
        # Selectors for compare at price (original price before discount)
        compare_price_elem = (
            card.select_one(".price.price--compare") or
            card.select_one(".price-item.price-item--compare")
        )

        compare_raw = compare_price_elem.get_text(strip=True) if compare_price_elem else "N/A"

        # Clean compare price
        compare_price_clean = "N/A"
        if compare_raw != "N/A":
            num_compare = re.sub(r"[^0-9.]", "", compare_raw)
            try:
                compare_price_clean = float(num_compare)
            except:
                compare_price_clean = "N/A"

       # ===== INVENTORY =====
        inventory_elem = (
            card.select_one(".product-item__inventory") or
            card.select_one(".inventory") or
            card.select_one(".stock") or
            card.select_one(".product-stock") or
            None
        )

        inventory_raw = inventory_elem.get_text(strip=True) if inventory_elem else "N/A"

        # Clean inventory
        inventory_clean = "N/A"
        if inventory_raw != "N/A":
            # 只提取数字，例如 “In stock (24)” → 24
            num3 = re.sub(r"[^0-9]", "", inventory_raw)
            if num3.isdigit():
                inventory_clean = int(num3)
            else:
                inventory_clean = inventory_raw

        # Save record
        all_products.append({
        "Page": page,
        "Product_Name": name,
        "Price_Raw": price_raw,
        "Price_Clean": price_clean,
        "Compare_Price_Raw": compare_raw,
        "Compare_Price_Clean": compare_price_clean,
        "Inventory_Raw": inventory_raw,
        "Inventory_Clean": inventory_clean
        })

    page += 1
    time.sleep(1)

# Create CSV
df = pd.DataFrame(all_products)
df.to_csv("shopify_sales_products.csv", index=False)
df.head()


Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=1
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=2
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=3
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=4
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=5
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=6
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=7
Fetching: https://warehouse-theme-metal.myshopify.com/collections/audio?page=8
No more products found. Stopping.


,Page,Product_Name,Price_Raw,Price_Clean,Compare_Price_Raw,Compare_Price_Clean,Inventory_Raw,Inventory_Clean
0,1,JBL Flip 4 Waterproof Portable Bluetooth Speaker,Sale price$74.95,74.95,Regular price$99.95,99.95,"In stock, 672 units",672
1,1,Shure SRH440 Professional Studio Headphones,Sale price$99.00,99.00,N/A,N/A,"In stock, 65 units",65
2,1,AKG N20U Premium In-Ear Headphones,Sale price$129.95,129.95,N/A,N/A,"In stock, 141 units",141
3,1,Shure SB900A Lithium-Ion Rechargeable Battery,Sale price$118.00,118.00,N/A,N/A,"In stock, 148 units",148
4,1,Pioneer PL-990 Automatic Stereo Turntable,Sale price$179.00,179.00,N/A,N/A,"In stock, 148 units",148
